# Dia6 - Gerador de Dados Indian E-Commerce + Fato Gold

**Objetivo:** Gerar os arquivos CSV do dataset Indian E-Commerce (customers, products, sales) na Landing, criar a tabela fato Gold `fat_vendas` em modelo estrela (star schema) com chaves substitutas e foreign keys, e popular a fato via `MERGE` a partir de uma Silver intermediária.

| Item | Valor |
|---|---|
| **Camada** | Landing (geração) + Silver + Gold (fato) |
| **Entrada** | CSVs gerados sinteticamente em `/Volumes/{catalog}/landing/files/indian/` |
| **Saída** | `{catalog}.silver.fat_vendas`, `{catalog}.gold.fat_vendas` |
| **Ordem no pipeline** | Antes de `Dia5-Create-Tables-Indian` (que cria as dimensões Gold) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Geração dos CSVs do dataset Indian E-Commerce

Cria a pasta no Volume e grava os 3 CSVs (customers, products, sales) com dados fictícios de e-commerce indiano.

In [0]:
dbutils.fs.mkdirs(f"/Volumes/{catalog}/landing/files/indian/")

customers_csv = """Customer_ID,Customer_Name,Gender,Date_of_Birth,Email,Phone,Registration_Date,Customer_Tier
CUST001,Aarav Sharma,male,1990-03-12,aarav.sharma@example.com,9876500001,2023-01-15,Gold
CUST002,Priya Patel,female,1988-07-22,priya.patel@example.com,9876500002,2023-02-10,Silver
CUST003,Rohan Mehta,male,1995-11-05,rohan.mehta@example.com,9876500003,2023-03-18,Bronze
CUST004,Sneha Reddy,female,1992-01-30,sneha.reddy@example.com,9876500004,2023-04-02,Gold
CUST005,Vikram Nair,male,1985-09-14,vikram.nair@example.com,9876500005,2023-05-20,Silver
CUST006,Ananya Iyer,female,1998-06-08,ananya.iyer@example.com,9876500006,2023-06-11,Bronze
CUST007,Karan Malhotra,male,1993-12-25,karan.malhotra@example.com,9876500007,2023-07-07,Gold
CUST008,Divya Singh,female,1991-04-17,divya.singh@example.com,9876500008,2023-08-19,Silver
"""
dbutils.fs.put(f"/Volumes/{catalog}/landing/files/indian/customers.csv", customers_csv, overwrite=True)

products_csv = """Product_ID,Product_Name,Category,Brand
PROD001,Wireless Earbuds,Electronics,SoundWave
PROD002,Cotton T-Shirt,Apparel,UrbanStyle
PROD003,Stainless Steel Bottle,Home,HydroLife
PROD004,Running Shoes,Footwear,SpeedFit
PROD005,Yoga Mat,Fitness,FlexPro
PROD006,Smartwatch,Electronics,TechPulse
PROD007,Backpack,Accessories,TrailGear
PROD008,Desk Lamp,Home,BrightHome
"""
dbutils.fs.put(f"/Volumes/{catalog}/landing/files/indian/products.csv", products_csv, overwrite=True)

sales_csv = """Order_ID,Order_Date,Order_Time,Customer_ID,Product_ID,Quantity,Unit_Price,Order_Value,Shipping_Cost,Coupon_Discount,Total_Amount,Order_Status
ORD1001,2026-09-01,10:15:00,CUST001,PROD001,2,49.90,99.80,5.00,0.00,104.80,Delivered
ORD1002,2026-09-02,14:30:00,CUST002,PROD004,1,89.90,89.90,7.50,10.00,87.40,Delivered
ORD1003,2026-09-03,09:45:00,CUST003,PROD002,3,19.90,59.70,4.00,0.00,63.70,Delivered
ORD1004,2026-09-04,16:20:00,CUST004,PROD006,1,179.90,179.90,0.00,0.00,179.90,Delivered
ORD1005,2026-09-05,13:10:00,CUST005,PROD003,2,29.90,59.80,0.00,15.00,44.80,Delivered
ORD1006,2026-09-06,13:10:00,CUST001,PROD005,1,39.90,39.90,4.50,0.00,44.40,Delivered
ORD1007,2026-09-07,17:35:00,CUST006,PROD007,1,59.90,59.90,0.00,0.00,59.90,Delivered
ORD1008,2026-09-08,08:50:00,CUST007,PROD008,2,34.90,69.80,4.00,0.00,73.80,Delivered
ORD1009,2026-09-09,12:15:00,CUST008,PROD006,1,49.90,49.90,0.00,0.00,49.90,Cancelled
ORD1010,2026-09-10,12:05:00,CUST002,PROD006,1,184.90,184.90,0.00,0.00,184.90,Delivered
"""
dbutils.fs.put(f"/Volumes/{catalog}/landing/files/indian/sales.csv", sales_csv, overwrite=True)

## Gold — DDL da tabela fato `fat_vendas` (star schema)

Cria a tabela fato com chaves substitutas (SK) para as 4 dimensões e adiciona foreign keys constraint para integridade referencial. `CREATE OR REPLACE TABLE` garante idempotência da DDL.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`gold`.`fat_vendas` ( 
	ven_sk_data_venda BIGINT NOT NULL COMMENT 'SK da data da venda',
	ven_sk_hora SMALLINT NOT NULL COMMENT 'SK hora da venda',
	ven_sk_produto BIGINT NOT NULL COMMENT 'SK do produto',
	ven_sk_cliente BIGINT NOT NULL COMMENT 'SK do cliente',
	ven_id_venda STRING NOT NULL COMMENT 'ID da venda',
	ven_qt_itens SMALLINT NOT NULL COMMENT 'Quantidade de itens da venda',
	ven_vl_unitario FLOAT NOT NULL COMMENT 'Valor unitário da venda',
	ven_vl_total FLOAT NOT NULL COMMENT 'Valor total da venda',
	ven_vl_frete FLOAT NOT NULL COMMENT 'Valor do frete',
	ven_vl_desconto FLOAT NOT NULL COMMENT 'Valor do desconto na venda',
	ven_vl_total_pago FLOAT NOT NULL COMMENT 'Valor total pago na venda',
	ven_dh_atualizacao TIMESTAMP NOT NULL DEFAULT from_utc_timestamp(now(), 'GMT-3') COMMENT 'Data/hora da última atualização da linha',
	    CONSTRAINT pk_fat_vendas PRIMARY KEY ( ven_sk_data_venda, ven_sk_hora, ven_sk_produto, ven_sk_cliente )
 ) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported',
                    'delta.logRetentionDuration'='7 days',
					'delta.autoOptimize.autoCompact'='auto',
                    'spark.databricks.delta.autoCompact.enabled'='true');

COMMENT ON TABLE `${catalog}`.`gold`.`fat_vendas` IS 'A tabela [fat_vendas] contém dados sobre as vendas do Indian E-Commerce Sales.';

ALTER TABLE `${catalog}`.`gold`.`fat_vendas` ADD CONSTRAINT fk_fat_vendas_dim_datas FOREIGN KEY ( ven_sk_data_venda ) REFERENCES `${catalog}`.`gold`.`dim_datas` ( dat_sk_data ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE `${catalog}`.`gold`.`fat_vendas` ADD CONSTRAINT fk_fat_vendas_dim_horas FOREIGN KEY ( ven_sk_hora ) REFERENCES `${catalog}`.`gold`.`dim_horas` ( hor_sk_hora ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE `${catalog}`.`gold`.`fat_vendas` ADD CONSTRAINT fk_fat_vendas_dim_produtos FOREIGN KEY ( ven_sk_produto ) REFERENCES `${catalog}`.`gold`.`dim_produtos` ( pro_sk_produto ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE `${catalog}`.`gold`.`fat_vendas` ADD CONSTRAINT fk_fat_vendas_dim_clientes FOREIGN KEY ( ven_sk_cliente ) REFERENCES `${catalog}`.`gold`.`dim_clientes` ( cli_sk_cliente ) ON DELETE NO ACTION ON UPDATE NO ACTION;

## Silver — Limpeza e preparação da fato

Cria a Silver `fat_vendas` a partir de `bronze.sales`, filtrando apenas pedidos `Delivered` com `Total_Amount > 0`. Converte IDs de negócio para o formato que permitirá o JOIN com as dimensões Gold.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`silver`.`fat_vendas` AS
  SELECT
      DATE_FORMAT(Order_Date, 'yyyyMMdd') AS ven_sk_data_venda,
      REPLACE(LEFT(Order_Time,5),':','') AS ven_id_hora,
      TRIM(Product_ID) AS ven_id_produto,
      TRIM(Customer_ID) AS ven_id_cliente,
      TRIM(Order_ID) AS ven_id_venda,
      Quantity AS ven_qt_itens,
      Unit_Price AS ven_vl_unitario,
      Order_Value AS ven_vl_total,
      Shipping_Cost AS ven_vl_frete,
      Coupon_Discount AS ven_vl_desconto,
      Total_Amount AS ven_vl_total_pago,
      from_utc_timestamp(now(), 'GMT-3') AS ven_dh_atualizacao
  FROM `${catalog}`.`bronze`.`sales`
  WHERE 1=1
    AND Order_Status = 'Delivered'
    AND Total_Amount > 0.0;

## MERGE — População da fato Gold com chaves substitutas

Faz `MERGE INTO` da Silver para a Gold, trocando IDs de negócio (texto) pelas chaves substitutas numéricas (SK) das dimensões via `LEFT JOIN`. `MERGE` é idempotente: atualiza vendas existentes e insere novas.

In [0]:
%sql
MERGE INTO `${catalog}`.`gold`.`fat_vendas` AS target
    USING (
        SELECT
            fat.ven_sk_data_venda,
            hor.hor_sk_hora AS ven_sk_hora,
            pro.pro_sk_produto AS ven_sk_produto,
            cli.cli_sk_cliente AS ven_sk_cliente,
            fat.ven_id_venda,
            fat.ven_qt_itens,
            fat.ven_vl_unitario,
            fat.ven_vl_total,
            fat.ven_vl_frete,
            fat.ven_vl_desconto,
            fat.ven_vl_total_pago
        FROM
            `${catalog}`.`silver`.`fat_vendas` AS fat
                LEFT JOIN `${catalog}`.`gold`.`dim_horas` AS hor
                    ON (fat.ven_id_hora = hor.hor_id_hora)
                LEFT JOIN `${catalog}`.`gold`.`dim_produtos` AS pro
                    ON (fat.ven_id_produto = pro.pro_id_produto)
                LEFT JOIN `${catalog}`.`gold`.`dim_clientes` AS cli
                    ON (fat.ven_id_cliente = cli.cli_id_cliente)
    ) AS source
        ON (target.ven_sk_data_venda = source.ven_sk_data_venda AND target.ven_sk_hora = source.ven_sk_hora
            AND target.ven_sk_produto = source.ven_sk_produto AND target.ven_sk_cliente = source.ven_sk_cliente
            AND target.ven_id_venda = source.ven_id_venda)
WHEN MATCHED THEN
    UPDATE SET
        ven_qt_itens        = source.ven_qt_itens,
        ven_vl_unitario     = source.ven_vl_unitario,
        ven_vl_total        = source.ven_vl_total,
        ven_vl_frete        = source.ven_vl_frete,
        ven_vl_desconto     = source.ven_vl_desconto,
        ven_vl_total_pago   = source.ven_vl_total_pago,
        ven_dh_atualizacao  = from_utc_timestamp(now(), 'GMT-3')
WHEN NOT MATCHED THEN
    INSERT (ven_sk_data_venda, ven_sk_hora, ven_sk_produto, ven_sk_cliente, ven_id_venda, ven_qt_itens, ven_vl_unitario, ven_vl_total, ven_vl_frete, ven_vl_desconto, ven_vl_total_pago)
    VALUES (source.ven_sk_data_venda, source.ven_sk_hora, source.ven_sk_produto, source.ven_sk_cliente, source.ven_id_venda, source.ven_qt_itens, source.ven_vl_unitario, source.ven_vl_total, source.ven_vl_frete, source.ven_vl_desconto, source.ven_vl_total_pago);